In [1]:
import asyncio
import logging
from ipaddress import IPv4Address

from scylla.batch import Batch
from scylla.enums import Consistency
from scylla.execution_profile import ExecutionProfile
from scylla.policies import HostFilter, Peer
from scylla.session_builder import SessionBuilder
from scylla.statement import Statement

logger = logging.getLogger(__name__)

In [2]:
class CustomHostFilter(HostFilter):
    def accept(self, peer: Peer) -> bool:
        peer_addr_ipv4 = peer.address[0]

        assert isinstance(peer_addr_ipv4, IPv4Address)

        # Accept only loopback nodes for this demo
        return peer_addr_ipv4.is_loopback

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
)


In [3]:
session = await (
    SessionBuilder()
    .contact_points([("127.0.0.2", 9042)])
    .host_filter(CustomHostFilter())
    .connect()
)

print("Connected to local Scylla cluster")

2026-05-26 13:05:30,398 [INFO] scylla.cluster.worker: Node added to cluster: 01fab70d-f8d2-4c11-9c1c-b84afadd2288 - 127.0.0.2:9042
2026-05-26 13:05:30,399 [INFO] scylla.cluster.worker: Node added to cluster: cf382e10-e0a6-4082-80ac-6f684076ec8e - 127.0.0.3:9042
2026-05-26 13:05:30,400 [INFO] scylla.cluster.worker: Node added to cluster: fbdef9fe-f7a0-4a31-bfe3-be83bb8d16c4 - 127.0.0.4:9042


Connected to local Scylla cluster


In [4]:
table_name = "example_table"
schema = "id int PRIMARY KEY, value text"

await session.execute("""
CREATE KEYSPACE IF NOT EXISTS example_ks
WITH replication = {
    'class': 'NetworkTopologyStrategy',
    'replication_factor': 3
};
""")

await session.execute("USE example_ks;")

await session.execute(f"DROP TABLE IF EXISTS {table_name};")

await session.execute(
    f"""
    CREATE TABLE IF NOT EXISTS {table_name} (
        {schema}
    );
    """
)

print(f"Table '{table_name}' ready")

Table 'example_table' ready


In [5]:
prepared_insert = await session.prepare(
    Statement(
        f"INSERT INTO {table_name} (id, value) VALUES (?, ?)"
    )
)

prepared_insert = prepared_insert.with_execution_profile(
    ExecutionProfile(
        timeout=10,
        consistency=Consistency.All,
    )
)

print("Prepared insert statement created")

Prepared insert statement created


In [6]:
values = [
    (0, "first natural number"),
    (1, "identity element of group Z with * as multiplication"),
    (2, "almost e"),
]

await asyncio.gather(
    *[session.execute(prepared_insert, value) for value in values]
)

print("Async inserts completed")

Async inserts completed


In [7]:
batch_values = [
    (3, "almost pi"),
    (4, "composite"),
    (5, "prime"),
]

batch_statement = Batch()

batch_statement.add_all(
    items=[
        (prepared_insert, value)
        for value in batch_values
    ]
)

await session.batch(batch_statement)

print("Batch insert completed")

Batch insert completed


In [8]:
prepared_select = (
    await session.prepare(
        f"SELECT * FROM {table_name}"
    )
)

prepared_select = (
    prepared_select
    .with_page_size(1)
    .with_consistency(Consistency.One)
)

print("Prepared select statement created")

Prepared select statement created


In [9]:
page_result = await session.execute(prepared_select)

print("Manual paging")

while page_result:
    print("\n================ PAGE START ================\n")

    for row in page_result.iter_current_page():
        print(row)

    print("\n================ PAGE END ==================\n")

    page_result = await page_result.fetch_next_page()

Manual paging

================ PAGE START ================

{'id': 5, 'value': 'prime'}

================ PAGE END ==================


================ PAGE START ================

{'id': 1, 'value': 'identity element of group Z with * as multiplication'}

================ PAGE END ==================


================ PAGE START ================

{'id': 0, 'value': 'first natural number'}

================ PAGE END ==================


================ PAGE START ================

{'id': 2, 'value': 'almost e'}

================ PAGE END ==================


================ PAGE START ================

{'id': 4, 'value': 'composite'}

================ PAGE END ==================


================ PAGE START ================

{'id': 3, 'value': 'almost pi'}

================ PAGE END ==================



In [10]:
result = await session.execute(prepared_select)

print("Async paging")

async for row in result:
    print(row)

Async paging
{'id': 5, 'value': 'prime'}
{'id': 1, 'value': 'identity element of group Z with * as multiplication'}
{'id': 0, 'value': 'first natural number'}
{'id': 2, 'value': 'almost e'}
{'id': 4, 'value': 'composite'}
{'id': 3, 'value': 'almost pi'}
